# Servidor

In [1]:
from typing import List, Callable, Any
import numpy as np
def sigmoidea(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def devSigmoidea(x: np.ndarray) -> np.ndarray:
    s = sigmoidea(x)
    return s * (1.0 - s)

def relu(x: np.ndarray) -> np.ndarray:
    return np.maximum(x, 0.0)

def devRelu(x: np.ndarray) -> np.ndarray:
    return np.where(x > 0, 1.0, 0.0)

def softmax(x: np.ndarray) -> np.ndarray:
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

def devSoftmax(x: np.ndarray) -> np.ndarray:
    s = softmax(x)
    s_vec = s.reshape(-1)
    jacobian_matrix = np.diag(s_vec) - np.outer(s_vec, s_vec)
    return jacobian_matrix

def mse(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    return np.mean((predicted - actually) ** 2)

def devMse(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    n = predicted.size
    return np.where(n > 0, (2.0 / n) * (predicted - actually), np.zeros_like(predicted))

def lostEntropy(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    eps = 1e-7
    p = np.clip(predicted, eps, 1.0 - eps)
    return -np.sum(actually * np.log(p))

def devLostEntropy(predicted: np.ndarray, actually: np.ndarray) -> np.ndarray:
    eps = 1e-7
    p = np.clip(predicted, eps, 1.0 - eps)
    return -(actually / p)

In [2]:
class Layer:
    def __init__(
            self,
            neurons: int,
            activation: Callable[[np.ndarray], np.ndarray],
            derivada: Callable[[np.ndarray], np.ndarray],
            name: str):
        self.neurons: int = neurons
        self.activacion = activation
        self.derivada = derivada
        self.name: str = name
        
    def export(self)->str:
        return f"{self.name}-{self.neurons}-{self.activacion.__name__}-{self.derivada.__name__}"
    
    @staticmethod
    def load(layer: str)->'Layer':
        name, neurons_str, activationName, derivadaName = layer.split("-")
        neurons = int(neurons_str)
        activation = None
        derivada = None
        match(activationName):
            case "sigmoidea":
                activation = sigmoidea
            case "relu":
                activation = relu
            case "softmax":
                activation = softmax
        
        match(derivadaName):
            case "devSigmoidea":
                derivada = devSigmoidea
            case "devRelu":
                derivada = devRelu
            case "devSoftmax":
                derivada = devSoftmax
        return Layer(neurons, activation, derivada, name)
        

class Model:
    def __init__(
            self,
            sequential: Any,
            w: List[np.ndarray],
            b: List[np.ndarray]):
        self.sequential = sequential
        self.set_parameters(w, b)
        
    def set_parameters(self, w: List[np.ndarray], b: List[np.ndarray]):
        self.w = [np.array(weights, dtype=np.float32) for weights in w]
        self.b = [np.array(bias, dtype=np.float32) for bias in b]
        
    def getParams(self):
        return (self.w, self.b)

    def fordward(self, x: np.ndarray) -> np.ndarray:
      neu = [None] * len(self.sequential)
      z = [None] * len(self.sequential)
      neu[0] = x
      for i in range(1, len(self.sequential)):
        neu[i] = self.sequential[i].activacion(np.dot(neu[i - 1], self.w[i - 1]) + self.b[i])
      return neu[-1]

class Sequential:
    def __init__(self, *layers: Layer):
        self.layers = layers

    def __len__(self):
        return len(self.layers)

    def __getitem__(self, i):
        return self.layers[i]
    
    def export(self):
        export = ""
        nLayer = len(self.layers)
        for index, i in enumerate(self.layers):
            if index < nLayer - 1:
                export += f"{i.export()}\n"
            else:
                export += f"{i.export()}"
        return export
    
    @staticmethod
    def load(layers: str)->'Sequential':
        internal = []
        for i in layers.split("\n"):
            internal.append(Layer.load(i))
        return Sequential(internal)


In [ ]:
from socket import socket
from typing import Dict, Any
import time
import pickle
import json
import os

def recvall(sock: socket, n: int) -> bytearray:
    data = bytearray()
    while len(data) < n:
        packet = sock.recv(n - len(data))
        if not packet:
            raise ConnectionError("Conexión cerrada antes de recibir todos los datos esperados")
        data.extend(packet)
    return data

class State:
    
    def __init__(self, params: Dict[str, Any]):
        self.data = params
    
    def do(self, sock: socket):
        pass
    
    def next(self):
        return HandShake(self.data)

class HandShake(State):
    
    def do(self, sock: socket):
        workers = self.data["workers"]
        workersList = []
        # self.data["history"].append({})
        self.data["startTime"] = time.time_ns()
        params = {
            "sequential": self.data["sequential"].export(), 
            "devError": self.data["devError"],
            "dsName": self.data["dsName"],
            "split": self.data["split"],
            "shard": self.data["shard"],
            "shardPosition": 0,
            "batchSize": self.data["batchSize"],
            "labels": ["image", "label"],
            "seed": self.data["seed"],
            "token": os.getenv("token")
        }
        overheadStart = time.time_ns()
        while workers != 0:
            conn, address = sock.accept()
            print(f"se ha conectado {address}")
            workers-=1
            workersList.append(conn)
            params["shardPosition"] = workers
            json_data = json.dumps(params).encode("utf-8")
            conn.sendall(len(json_data).to_bytes(8, 'big'))
            conn.sendall(json_data)
        self.data["overheadTime"] = time.time_ns() - overheadStart
        self.data["workersList"] = workersList
                
    def next(self):
        return Recolection(self.data)
    
class Recolection(State):
    
    def do(self, sock: socket):
        try:
            workers = len(self.data["workersList"])
            trueList = []
            for i in range(workers):
                worker_socket = self.data["workersList"][i]
                ans = worker_socket.recv(10).decode("utf-8")
                cod, workerNumber = ans.split("-")
                if(cod == "y"):
                    trueList.append(self.data["workersList"][int(workerNumber)])
                    continue
                self.data["workersList"][i].close()
            self.data["workersList"] = trueList
        except Exception as e:
            print(e)
                
    def next(self):
        return TrainBatch(self.data)

class TrainBatch(State):
    
    def do(self, sock: socket):
        try:
            self.data["epochs"] -= 1
            wb_data = pickle.dumps((self.data["w"], self.data["b"]))
            for conn in self.data["workersList"]:
                conn.sendall(len(wb_data).to_bytes(8, 'big'))
                conn.sendall(wb_data)
        except Exception as e:
            pass
                
    def next(self):
        return End(self.data)

class End(State):
    
    def do(self, sock: socket):
        try:
            sequential = self.data["sequential"]
            learningRate = self.data["learningRate"]
            batch = self.data["batch"]
            harvestStart = time.time_ns()
            for conn in self.data["workersList"]:
                ans = conn.recv(10).decode("utf-8")
                cod, workerNumber = ans.split("-")
                if cod == "y":
                    length_prefix = recvall(conn, 8)
                    message_length = int.from_bytes(length_prefix, 'big')
                    data = recvall(conn, message_length)
                    w_grad_batch, b_grad_batch = pickle.loads(data)
                    for i in range(len(sequential) - 1):
                        self.data["w"][i] -= learningRate * (w_grad_batch[i] / batch)
                        self.data["b"][i+1] -= learningRate * (b_grad_batch[i+1] / batch)
            self.data["harvestTime"] = time.time_ns() - harvestStart
        except Exception as e:
            print(e)
            
    def next(self):
        if self.data["epochs"] > 0:
            for conn in self.data["workersList"]:
                conn.sendall(b"y")
                self.data["history"].append({
                    "epoch": epoch,
                    "batch": a,
                    "loss_avg": np.mean(batch_losses),
                    "accuracy": accuracy,
                    "test_accuracy": test_accuracy,
                    "cpu_percent":0,
                    "ram":0,
                    "time": (time.time_ns()-self.data["startTime"])/1000,
                    "overheadTime":self.data["overheadTime"]/1000,
                    "harvestTime": self.data["harvestTime"]/1000,
                    "sumTime": (end - sumStart)/1000
                })
            return TrainBatch(self.data)
        for conn in self.data["workersList"]:
            conn.sendall(b"n")
        return None

In [4]:
from typing import Callable
from multiprocessing import Process, Pipe
import psutil
from tqdm import tqdm
import os
import traceback
import time
import math
import socket
import json
import pickle
import socket

def __forward(neu, x_sample: np.ndarray, z, sequential, w, b):
    neu[0] = x_sample
    for i in range(1, len(sequential)):
        # print(f"w[{i}] = ", w[i - 1].shape)
        z[i] = np.dot(neu[i - 1], w[i - 1]) + b[i]
        neu[i] = sequential[i].activacion(z[i])

def __evaluateBatch(w, b, x, y, sequential, error, son):
    correct_predictions = 0
    errors = []
    for x_b, y_b in zip(x, y):
        neu_v = [None] * len(sequential)
        z_v = [None] * len(sequential)
        __forward(neu_v, x_b, z_v, sequential, w, b)
        errors.append(error(neu_v[-1], y_b))
        if np.argmax(neu_v[-1]) == np.argmax(y_b):
            correct_predictions += 1
    son.send((correct_predictions, errors))
    son.close()

def __evaluate(w, b, x_test, y_test, sequential, error):
    maxProcess = int(os.getenv("NUM_PROCESS", 1))
    batchForProcess = max(1, x_test.shape[0] // maxProcess)
    correct_predictions = 0
    process = []
    errors = []
    for i in range(maxProcess):
        father, son = Pipe()
        slace_bottom = i * batchForProcess
        if i == maxProcess - 1:
            slace_top = x_test.shape[0] 
        else:
            slace_top = (i+1) * batchForProcess
        x_batch = x_test[slace_bottom : slace_top]
        y_batch = y_test[slace_bottom : slace_top]
        p = Process(target=__evaluateBatch, args=(w, b, x_batch, y_batch, sequential, error, son))
        p.start()
        process.append((p, father))
    for p, conection in process:
        n, e = conection.recv()
        errors.append(e)
        correct_predictions += n
        p.join()
    return (correct_predictions/max(x_test.shape[0], 1), errors)

def __backward(dEdz, z, sequential, w) -> np.ndarray:
    for i in range(len(sequential) - 2, 0, -1):
        dEdz[i] = (dEdz[i+1] @ w[i].T) * sequential[i].derivada(z[i])

def __batch(x_b_batch, y_b_batch, w, b, sequential, devError, conn):
    try:
        w_grad_sample_list = [np.zeros_like(wi) for wi in w]
        b_grad_sample_list = [np.zeros_like(bi) for bi in b]
        for x_b, y_b in zip(x_b_batch, y_b_batch):
            neu = [None] * len(sequential)
            z = [None] * len(sequential)
            __forward(neu, x_b, z, sequential, w, b)
            dEdz = [None] * len(sequential)
            de = devError(neu[-1], y_b)
            if de.shape == (1,):
                dEdz[-1] = de * sequential[-1].derivada(z[-1])
            else:
                dEdz[-1] = de @ sequential[-1].derivada(z[-1])
            __backward(dEdz, z, sequential, w)
            for i in range(len(sequential) - 1):
                w_grad_sample_list[i] += np.outer(neu[i], dEdz[i+1])
                b_grad_sample_list[i+1] += dEdz[i+1]
        conn.send(("OK", (w_grad_sample_list, b_grad_sample_list)))
    except Exception as e:
        error_msg = traceback.format_exc()
        conn.send(("ERROR", error_msg))
    finally:
        conn.close()

def __oldBatch(x_b, y_b, w, b, w_grad_batch, b_grad_batch, sequential, devError):
    neu = [None] * len(sequential)
    z = [None] * len(sequential)
    __forward(neu, x_b, z, sequential, w, b)
    dEdz = [None] * len(sequential)
    de = devError(neu[-1], y_b)
    if de.shape == (1,):
      dEdz[-1] = de * sequential[-1].derivada(z[-1])
    else:
      dEdz[-1] = de @ sequential[-1].derivada(z[-1])
    __backward(dEdz, z, sequential, w)
    for i in range(len(sequential) - 1):
        w_grad_sample = np.outer(neu[i], dEdz[i+1])
        w_grad_batch[i] += w_grad_sample
        b_grad_batch[i+1] += dEdz[i+1]

def fit(
    dataset: str,
    epochs: int,
    learningRate: float,
    sequential,
    error: Callable,
    devError: Callable,
    InitB: float,
    batch: int = 1,
    workers: int = 1,
    datasetPorcent: int = 1,
    verbose:bool = False,
    test: bool = False,
    x_test: np.ndarray = np.zeros((1)),
    y_test: np.ndarray = np.zeros((1)),
):
    os.environ.pop("OMP_NUM_THREADS", None) 
    os.environ.pop("OPENBLAS_NUM_THREADS", None) 
    os.environ.pop("MKL_NUM_THREADS", None)
    os.environ.pop("VECLIB_MAXIMUM_THREADS", None)
    os.environ.pop("NUMEXPR_NUM_THREADS", None)
    w = []
    b = []
    history = []
    seed = int(os.getenv("seed", 1))
    for i in range(len(sequential)):
        if i < len(sequential) - 1:
            shape = (sequential[i].neurons, sequential[i+1].neurons)
            limit = np.sqrt(6 / (shape[0] + shape[1]))
            w.append(np.random.uniform(-limit, limit, size=shape))
        b.append(np.full((sequential[i].neurons,), InitB))
    estado_actual = State({
        "sequential": sequential, 
        "devError": devError.__name__,
        "dsName": dataset,
        "split": "train",
        "shard": int((1/datasetPorcent) * batch * workers),
        "workers":workers,
        "shardPosition": 0,
        "batchSize": batch,
        "batch": batch,
        "seed": seed,
        "labels": ["image", "label"],
        "w": w,
        "b": b,
        "epochs": epochs,
        "learningRate": learningRate,
        "verbose": verbose,
        "history": history
    })
    HOST = "127.0.0.1"
    PORT = 65432
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind((HOST, PORT))
        s.listen()
        while estado_actual is not None:
            estado_actual.do(s)
            estado_actual = estado_actual.next()
    return (Model(sequential, w, b), history)

In [5]:
(modelSerial, historyComplete) = fit(
    dataset="ILSVRC/imagenet-1k",
    epochs=1,
    learningRate=0.1,
    sequential=Sequential(
        Layer(50176, relu, devRelu, "input"),
        Layer(512, relu, devRelu, "hidden"),
        Layer(128, relu, devRelu, "hidden"),
        Layer(1000, softmax, devSoftmax, "output")
    ),
    error=lostEntropy,
    devError=devLostEntropy,
    InitB=0.15,
    batch=512,
    verbose=False,
    datasetPorcent=0.1
    # test=True,
    # x_test=x_test,
    # y_test=y_test
)

se ha conectado ('127.0.0.1', 43992)


In [6]:
from datasets import load_dataset, load_dataset_builder
import numpy as np
from typing import Tuple
import os
import tempfile
import math

def downloadDataset(dataset: str, splitName: str, split: Tuple[int, int]):
    sharp, index = split
    token = os.getenv("token")
    ds = load_dataset(dataset, split=splitName, token=token)
    builder = load_dataset_builder(dataset, token=token,)
    total = builder.info.splits[splitName].num_examples
    return (ds.shard(num_shards=sharp, index=index), (total + sharp - 1) // sharp)

def one_hot_encode(labels, num_classes=10):
    return np.eye(num_classes)[labels]

def getBatch(ds, batchSize:int, labels: Tuple[str, str], size: int, shard: int, shape=(224,224), classNumber = 1000):
    newShape = shape[0]*shape[1]
    x = np.zeros((batchSize, newShape))
    y = np.zeros((batchSize, classNumber))
    try:
        img, label = labels
        folder = os.path.join('.', f'data-{batchSize}')
        xFolder = os.path.join(folder, 'x')
        yFolder = os.path.join(folder, 'y')
        if not os.path.exists(folder):
            os.mkdir(folder)
            os.mkdir(xFolder)
            os.mkdir(yFolder)
        for i in range(math.ceil(size/batchSize)):
            path_x = os.path.join(xFolder, f"batch-{shard}-{i}.npy")
            path_y = os.path.join(yFolder, f"batch-{shard}-{i}.npy")
            if os.path.exists(path_x) and os.path.exists(path_y):
                x = np.load(path_x)
                y = np.load(path_y)
                yield (x, y)
                continue
            for index, element in enumerate(ds.take(batchSize)):
                image = element[img].resize(shape)
                image_label = element[label]
                j = index%batchSize
                image = np.array(image, dtype=np.float32)
                if image.ndim == 3 and image.shape[2] >= 3:
                    image = (
                        image[:, :, 0] * 0.299 +
                        image[:, :, 1] * 0.587 +
                        image[:, :, 2] * 0.114
                    ) / 255.0
                elif image.ndim == 2:
                    image = image / 255.0
                else:
                    image = np.mean(image, axis=2) / 255.0 if image.ndim == 3 else image / 255.0
                x[j] = image.reshape((newShape))
                y[j] = one_hot_encode(image_label, classNumber)
            np.save(path_x, x)
            np.save(path_y, y)
            yield (x, y)
            x = np.zeros((batchSize, newShape))
            y = np.zeros((batchSize, classNumber))
    except Exception as e:
        print(e)
        yield (x, y)


/mnt/c/Users/Usuario UTP/Documents/tareas/redNeuronal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
import pandas as pd
import altair as alt
from sklearn.metrics import confusion_matrix
import io
import base64
from PIL import Image

def plot_loss(history):
    df_history = pd.DataFrame(history)
    df_epoch = df_history.groupby('epoch')['loss_avg'].mean().reset_index()
    line = alt.Chart(df_epoch).mark_line(color='#2196F3', strokeWidth=3).encode(
        x=alt.X('epoch:Q', title='Época', axis=alt.Axis(tickMinStep=1)),
        y=alt.Y('loss_avg:Q', title='Pérdida (loss_avg)', scale=alt.Scale(type='log')), # Escala logarítmica suele ser mejor para loss_avg
        tooltip=['epoch', 'loss_avg']
    )
    points = line.mark_point(size=60, filled=True).encode(
        opacity=alt.value(1)
    )
    return (line + points).properties(
        title='Progreso del Entrenamiento: loss_avg vs Epoch',
        width=600,
        height=300
    ).interactive()

def plot_confusion_matrix(m, x, y, label_names):
    y_pred = np.zeros_like(y)
    for i in range(len(x)):
        y_pred[i] = np.argmax(m.fordward(x[i]))
    cm = confusion_matrix(y, y_pred)
    data = []
    for i in range(len(cm)):
        for j in range(len(cm)):
            data.append({
                'Actual': str(label_names[i]),
                'Predicho': str(label_names[j]),
                'Cantidad': int(cm[i, j])
            })
    df_cm = pd.DataFrame(data)
    threshold = float(cm.max() / 2)
    base = alt.Chart(df_cm).encode(
        x=alt.X('Predicho:O', title='Clase Predicha', sort=list(label_names.values())),
        y=alt.Y('Actual:O', title='Clase Real', sort=list(label_names.values()))
    )
    heatmap = base.mark_rect().encode(
        color=alt.Color('Cantidad:Q', scale=alt.Scale(scheme='blues'), title='Frecuencia'),
        tooltip=['Actual', 'Predicho', 'Cantidad']
    )
    text = base.mark_text(baseline='middle').encode(
        text='Cantidad:Q',
        color=alt.condition(
            f"datum.Cantidad > {threshold}",
            alt.value('white'),
            alt.value('black')
        )
    )
    return (heatmap + text).properties(
        title='Matriz de Confusión (CIFAR-10)',
        width=500,
        height=500,
    ).configure_axis(
        labelFontSize=12,
        titleFontSize=14
    )

def plot_accuracy_only(history):
    df_history = pd.DataFrame(history)
    df_epoch = df_history.groupby('epoch')['accuracy'].mean().reset_index()
    line = alt.Chart(df_epoch).mark_line(color='#2196F3', strokeWidth=3).encode(
        x=alt.X('epoch:Q', title='Época', axis=alt.Axis(tickMinStep=1)),
        y=alt.Y('accuracy:Q', title='Pérdida (accuracy)', scale=alt.Scale(type='log')), # Escala logarítmica suele ser mejor para accuracy
        tooltip=['epoch', 'accuracy']
    )
    points = line.mark_point(size=60, filled=True).encode(
        opacity=alt.value(1)
    )
    return (line + points).properties(
        title='Progreso del Entrenamiento: accuracy vs Epoch',
        width=600,
        height=300
    ).interactive()

def plot_time(history):
    df_history = pd.DataFrame(history)
    df_history = df_history.rename(columns={'time': 'totalTime'})
    
    df_epoch = df_history.groupby('epoch').agg({
        'totalTime': 'mean',
        'overheadTime': 'mean',
        'harvestTime': 'mean',
        'sumTime': 'mean'
    }).reset_index()
    df_melted = df_epoch.melt(
        id_vars=['epoch'], 
        value_vars=['totalTime', 'overheadTime', 'harvestTime', 'sumTime'],
        var_name='Metrica', 
        value_name='ms'
    )
    nearest = alt.selection_point(nearest=True, on='mouseover', 
                                  fields=['epoch'], empty=False)
    lines = alt.Chart(df_melted).mark_line(strokeWidth=3).encode(
        x=alt.X('epoch:Q', title='Época'),
        y=alt.Y('ms:Q', title='Tiempo (ms)', scale=alt.Scale(type='symlog')), 
        color=alt.Color('Metrica:N', scale=alt.Scale(
            domain=['totalTime', 'overheadTime', 'harvestTime', 'sumTime'],
            range=['#2196F3', '#FF5722', '#000000', '#00913F']
        ))
    )
    selectors = alt.Chart(df_epoch).mark_rule(color='gray', strokeWidth=1).encode(
        x='epoch:Q',
        opacity=alt.condition(nearest, alt.value(0.3), alt.value(0)),
        tooltip=[
            alt.Tooltip('epoch:Q', title='Época'),
            alt.Tooltip('totalTime:Q', title='Tiempo Total (ms)', format='.2f'),
            alt.Tooltip('overheadTime:Q', title='Overhead (ms)', format='.2f'),
            alt.Tooltip('harvestTime:Q', title='Harvest (ms)', format='.2f'),
            alt.Tooltip('sumTime:Q', title='Sum (ms)', format='.2f'),
        ]
    ).add_params(nearest)
    points = lines.mark_point(size=80, filled=True).encode(
        opacity=alt.condition(nearest, alt.value(1), alt.value(0))
    )
    return (lines + selectors + points).properties(
        title='Análisis de Rendimiento',
        width=600,
        height=300,
    ).interactive()

def plot_wall_of_shame(m, x_test, y_test, label_names, num_errors=15):
    errors_found = []
    for i in range(len(x_test)):
        if len(errors_found) >= num_errors:
            break
        output = m.fordward(x_test[i])
        pred_idx = np.argmax(output)
        actual_idx = y_test[i]
        if pred_idx != actual_idx:
            img_array = (x_test[i].reshape((32, 32)) * 255).astype(np.uint8)
            img_pil = Image.fromarray(img_array)
            buffered = io.BytesIO()
            img_pil.save(buffered, format="PNG")
            img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            actual_name = label_names[actual_idx]
            pred_name = label_names[pred_idx]
            errors_found.append({
                "image": f"data:image/png;base64,{img_b64}",
                "Actual": actual_name,
                "Predicho": pred_name,
                "Leyenda": f"Real: {actual_name} | Pred: {pred_name}",
                "Confianza": float(np.max(output)),
                "row": len(errors_found) // 5,
                "col": len(errors_found) % 5
            })
    df_errors = pd.DataFrame(errors_found)
    base = alt.Chart(df_errors).encode(
        x=alt.X('col:O', axis=alt.Axis(labels=False, ticks=False, title=None)),
        y=alt.Y('row:O', axis=alt.Axis(labels=False, ticks=False, title=None))
    )
    image_layer = base.mark_image(width=70, height=70).encode(
        url='image',
        tooltip=['Actual', 'Predicho', 'Confianza']
    ).properties(
        title="Dataset Preview",
        width=600,
        height=400,
    )
    text_layer = base.mark_text(
        dy=55,
        fontSize=10,
        fontWeight='bold'
    ).encode(
        text='Leyenda:N',
        color=alt.value('#D32F2F')
    )
    return (image_layer + text_layer).properties(
        title="Wall of Shame: Errores del Modelo",
        padding=30
    ).configure_view(strokeWidth=0)
    
def plot_accuracy(history):
    df_history = pd.DataFrame(history)
    df_epoch = df_history.groupby('epoch').agg({
        'accuracy': 'mean',
        'test_accuracy': 'mean'
    }).reset_index()
    df_melted = df_epoch.melt(
        id_vars=['epoch'], 
        value_vars=['accuracy', 'test_accuracy'],
        var_name='Dataset', 
        value_name='Acc'
    )
    base = alt.Chart(df_melted).encode(
        x=alt.X('epoch:Q', title='Época', axis=alt.Axis(tickMinStep=1)),
        y=alt.Y('Acc:Q', title='Exactitud (Accuracy)', scale=alt.Scale(domain=[0, 1])),
        color=alt.Color('Dataset:N', scale=alt.Scale(
            domain=['accuracy', 'test_accuracy'],
            range=['#2196F3', '#FF9800']
        ), title='Leyenda')
    )
    lines = base.mark_line(strokeWidth=3)
    points = base.mark_point(size=60, filled=True)
    # selección más cercana para mostrar tooltips con ambos valores (train/test)
    nearest = alt.selection_point(nearest=True, on='mouseover', fields=['epoch'], empty='none')
    tooltip_rule = alt.Chart(df_epoch).mark_rule(color='gray').encode(
        x='epoch:Q',
        opacity=alt.condition(nearest, alt.value(0.3), alt.value(0)),
        tooltip=[
            alt.Tooltip('epoch:Q', title='Época'),
            alt.Tooltip('accuracy:Q', title='Train Acc', format='.2%'),
            alt.Tooltip('test_accuracy:Q', title='Test Acc', format='.2%')
        ]
    ).add_selection(nearest)
    return (lines + points + tooltip_rule).properties(
        title='Evolución del Accuracy: Entrenamiento vs. Prueba',
        width=600,
        height=300,
    ).interactive()


In [ ]:
import os
os.environ["token"] = os.getenv("HF_TOKEN", "")
(ds, size) = downloadDataset("ILSVRC/imagenet-1k", "train", (100000, 0))
x, y = next(getBatch(ds, batchSize, ("image", "label"), size, 100000))
plot_confusion_matrix(modelSerial, x, ds["test"]["label"], labelToMeaning).show()